<div align="center"> 

---



# **Cerebellum Mega Analysis**


<img src="https://static.wixstatic.com/media/da4857_5f6dbdb6bbd84328b1e31c7f2dafb673~mv2.png/v1/fill/w_224,h_226,al_c,q_85,usm_0.66_1.00_0.01,enc_auto/enigma_addiction-300x249.png" alt="image" style="width:300px;"/>


 </img>
</div>


---
# <u> **Project Description:**</u>

The aim of this study is to perform a multi-site mega/meta analysis to assess the effects of substance use on volumetric changes in cerebellar regions.

## **Primary Analysis: Matchit Cross-Validated Permutation Tests**

This notebook performs the main analysis using matched samples (propensity score matching) with cross-validated permutation-based significance testing.

## **Sensitivity Analyses Included:**
- **CC (Complete-Cases)**: Subjects with complete Age/Sex data
- **AI (Alternative-Imputation)**: Age/Sex imputed by Substance + Site
- **CA (Covariate-Adjusted)**: ComBat harmonization with stratified covariate residualization

See `03-2_Permutation_Variants.ipynb` for comprehensive comparison of all three approaches.

## **Analysis Details**
   1. Main effects: Acapulco cerebellar atlas
      - Volume ~ substance group (User vs HC, Substance vs HC)
   2. Cross-validation: Group-stratified k-fold (by Database)
   3. Statistical testing: Permutation tests (1,000 permutations)
   4. Effect size: Cohen's d
   5. Multiple testing: FDR correction

---

In [1]:
addTaskCallback(function(...) {set.seed(42);TRUE})
options(warn=-1) # to turn off warnings
#options(warn=0) # to turn on warnings
options(repr.plot.width=10, repr.plot.height=6) # to resize plots

1 
1

In [2]:
setwd("/mnt/g/Other computers/My Laptop/PhD/Psilantro/Enigma_cerebellum/Enigma_Cerebellum/")

## Load Packages

In [3]:
# Load required packages  
  pacman::p_load(tidyverse,devtools,readxl,ggbeeswarm,gtExtras,gt,webshot,webshot2,VIM,
                ggpubr,cowplot,gtsummary,ff,ggthemes,scales,mice,vtreat,sjmisc,SmartEDA,boot,MKinfer,MatchIt,
                ggcorrplot,olsrr,lme4,rstatix,janitor,coin,rcartocolor,ggdist,gghalves,ggnewscale,patchwork,
                lsr,tidyr,reshape2,broom,irr,ggpubr,caret,tidymodels,foreach,doParallel,optmatch,emmeans)

# NeuroCombat
 if (!require("neuroCombat")){
   devtools::install_github("jfortin1/neuroCombat_Rpackage") }


# Load NeuroCombat
library(neuroCombat)

Loading required package: neuroCombat



## Set settings for plotting

In [4]:
theme_output  <- theme(text = element_text(family = "Times New Roman"),
                        axis.text.x = element_text(size = 20),
                        axis.text.y = element_text(size = 20),
                        axis.title.x = element_text(size = 25, face = "bold"),
                        axis.title.y = element_text(size = 25, face = "bold"),
                        legend.text = element_text(size = 22), legend.title = element_text(size = 24),
                        title = element_text(size = 20,face = "italic"))

pal_sex <- c(alpha("#8E5A9C",1),alpha("#2879BC",1))
pal_user <- c(alpha("#757575",1),alpha("#059DC7",1))
pal_substances <- c("HC" = alpha("#757575",1), "ALC" = alpha("#C93F4D",1), 
    "ATS" = alpha("#EB8F33",1), "CANN" = alpha("#0B8755",1), 
    "COC" = alpha("#3A5589",1), "NIC" = alpha("#764EA1",1),
    "OPI" = alpha("#06AED5",1))

In [5]:
`%notin%` <- Negate(`%in%`)

## Load data

In [6]:
load("Data/RData/Volumes_Cerebellum.RData")

# **1 Analysis for Acapulco volumes**

In [7]:
num_perm <- 1000

## **1.1 Volume comparison of the full database**

### **1.1.1 Set data**

Final files:  
**Acapulco_volumes**  
**Acapulco_volumes_long** 

In [8]:
Acapulco_volumes <- Volumes_Cerebellum_Acapulco_Harmonized %>% 
                filter(Substance != "OPI") %>%
                mutate(group = if_else(Substance == "HC",0,1) %>%
                        factor(labels = c("HC","User")),
                        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","HC")), .before = 2) %>% 
                mutate(group = factor(group, levels = c("User","HC"))) %>%
                select(-c(Age,Sex,Control)) %>% unique() %>% 
                mutate(Database = factor(Database)) %>% 
                mutate(GM_Vol = Total_Cerebel_Vol-Corpus.Medullare,
                        Database = factor(Database))

Acapulco_cerebellar_regions_names <- Acapulco_volumes %>%
    select(contains(c(".","_Vol"))) %>% colnames()

# 1. transform the dataset to long format
Acapulco_volumes_long <- Acapulco_volumes %>% 
    pivot_longer(names_to = "region", values_to = "volume", Acapulco_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))

In [9]:
Acapulco_volumes %>% 
    select(Substance, group, ID_MRI, Database) %>%
    separate(ID_MRI, into = c("DB", "ID"), sep = "_", extra = "merge") %>%
    add_column(Acapulco_volumes %>% select(ID_MRI),.before = 3) %>% 
    write_csv("Data/Subjects.csv")

### **1.1.2 Users**

Final file: **p_values_Acapulco_user_vs_control**

#### Running the linear model

In [10]:
## Compute the group effect with a linear model (HC as reference)
Acapulco_lm_user_vs_control <- Acapulco_volumes_long %>%
  select(group, region, volume) %>%
  nest(data = -region) %>%
  mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
  mutate(test = map(data, ~ {
    m <- lm(volume ~ group, data = .)
    dat <- broom::tidy(m, conf.int = TRUE) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::mutate(
        df = df.residual(m),
        sigma = sigma(m))
    cont <- emmeans::emmeans(m, specs = "group") %>%
      emmeans::contrast(method = "pairwise")
    cont_df <- broom::tidy(cont)
    cohen_d <- cont_df$estimate / sigma(m)
    dat %>% mutate(effect.size = cohen_d)
  })) %>%
  unnest(test)


#### Obtain the null distributions

In [11]:
rm(Acapulco_nulls_user_vs_control)
# Check in null distribution matrix exists 
if(exists("nulls_user_vs_control")) {
    print(1)
} else {
    Acapulco_nulls_user_vs_control <- matrix(NA,ncol = Acapulco_volumes %>% 
        select(Acapulco_cerebellar_regions_names) %>% ncol(),nrow = num_perm)
}

In [12]:
Acapulco_compute_null_distributions <- function(Acapulco_nulls_user_vs_control) {
    if (sum(is.na(Acapulco_nulls_user_vs_control)) > 0) {
        cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
        on.exit(parallel::stopCluster(cl), add = TRUE)
        doParallel::registerDoParallel(cl)
        
        res <- foreach(i = seq_len(num_perm),
                       .packages = c("dplyr", "tidyr", "purrr", "broom"),
                       .export = c("Acapulco_volumes", "Acapulco_cerebellar_regions_names", "num_perm")) %dopar% {
            set.seed(42 + i)
            boot <- sample(seq_len(nrow(Acapulco_volumes)), size = nrow(Acapulco_volumes), replace = TRUE)
            
            x <- tibble(
                    group = stats::relevel(Acapulco_volumes$group[boot], ref = "HC")
                ) %>%
                dplyr::bind_cols(dplyr::select(Acapulco_volumes, dplyr::all_of(Acapulco_cerebellar_regions_names))) %>%
                tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                             names_to = "region",
                             values_to = "volume") %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .x)
                    broom::tidy(m) %>%
                        dplyr::filter(term == "groupUser") %>%
                        dplyr::select(statistic)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::pull(statistic)
            
            x
        }
        
        for (j in seq_len(num_perm)) {
            Acapulco_nulls_user_vs_control[j, ] <- res[[j]]
        }
    }
    return(Acapulco_nulls_user_vs_control)
}


#### Perform it

In [13]:
dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)

nulls_fp <- file.path("Statistics", "Linear_models", "Acapulco", "RData", "Acapulco_nulls_user_vs_control.RData")

if (!file.exists(nulls_fp)) {

    Acapulco_nulls_user_vs_control <- Acapulco_compute_null_distributions(Acapulco_nulls_user_vs_control)

    # add names to the nulls and transform to a tibble
    colnames(Acapulco_nulls_user_vs_control) <- Acapulco_lm_user_vs_control$region

    ci_lower <- apply(Acapulco_nulls_user_vs_control, 2, quantile, probs = 0.025)  
    ci_upper <- apply(Acapulco_nulls_user_vs_control, 2, quantile, probs = 0.975)

    Acapulco_lm_user_vs_control$perm_CI_lower <- ci_lower 
    Acapulco_lm_user_vs_control$perm_CI_upper <- ci_upper

    # # Calculate the p-values

    # Calculate permutation p-values aligned by region
    pvals <- purrr::map2_dbl(
        as.data.frame(Acapulco_nulls_user_vs_control),
        Acapulco_lm_user_vs_control$statistic, 
        ~ mean(abs(.x) >= abs(.y))
    )

    Acapulco_lm_user_vs_control_permuted <- Acapulco_lm_user_vs_control %>%
        dplyr::select(-data) %>%
        dplyr::mutate(
            p.value = pvals,
            perm_CI_lower = ci_lower[as.character(region)],
            perm_CI_upper = ci_upper[as.character(region)]
        )

    p_values_Acapulco_user_vs_control <- Acapulco_lm_user_vs_control_permuted %>%
        dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr"))

    save(Acapulco_nulls_user_vs_control,Acapulco_lm_user_vs_control_permuted,p_values_Acapulco_user_vs_control, file = nulls_fp)
} else {
    load(nulls_fp)
}

### **1.1.3 Substances**

Final file: **p_values_Acapulco_substance_vs_control**

#### Running the linear models

In [14]:
# Define a vector with the names of the substance
substances_names  <- unique(Acapulco_volumes$Substance)[unique(Acapulco_volumes$Substance) !="HC"] %>% as.vector()

In [15]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

Acapulco_lm_substance_vs_control <- map_df(substances_names, .f = ~ {
    Acapulco_volumes_long %>%
        dplyr::filter(Substance %in% c("HC", .x)) %>%
        dplyr::select(group, region, volume) %>%
        tidyr::nest(data = -region) %>%
        dplyr::mutate(data = purrr::map(data, ~ .x %>% dplyr::mutate(group = relevel(group, ref = "HC")))) %>%
        dplyr::mutate(test = purrr::map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% dplyr::mutate(effect.size = cohen_d)
        })) %>%
        tidyr::unnest(test) %>%
        dplyr::mutate(Substance = .x)
})

#### Obtain the null distributions

In [16]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions
if (!exists("Acapulco_nulls_substance_vs_control")){

Acapulco_nulls_substance_vs_control <- lapply(1:length(substances_names),
    matrix, data = NA,
    nrow = num_perm, 
    ncol = Acapulco_volumes %>% select(Acapulco_cerebellar_regions_names) %>% ncol()
    ) %>%
setNames(substances_names)
}


In [17]:
# Parallel permutation function (foreach) for substances vs control
Acapulco_compute_null_distributions_substances <- function(Acapulco_volumes,
                              Acapulco_cerebellar_regions_names,
                              substances_names,
                              Acapulco_nulls_substance_vs_control,
                              num_perm) {
  if (sum(is.na(unlist(Acapulco_nulls_substance_vs_control))) == 0) return(Acapulco_nulls_substance_vs_control)
  
  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)
  
  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr", "tidyr", "purrr", "broom")) %dopar% {
  # If this substance already has values, return as is
  current <- Acapulco_nulls_substance_vs_control[[i]]
  if (!is.null(current) && sum(is.na(current)) == 0) return(current)
  
  set.seed(42 + i)
  n_cols <- Acapulco_volumes %>% dplyr::select(dplyr::all_of(Acapulco_cerebellar_regions_names)) %>% ncol()
  nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)
  
  sub_df <- Acapulco_volumes %>%
    dplyr::filter(Substance %in% c("HC", substances_names[i])) %>%
    dplyr::select(Substance, group, dplyr::all_of(Acapulco_cerebellar_regions_names))
  
  for (j in seq_len(num_perm)) {
    boot <- sample.int(n = nrow(sub_df), replace = TRUE)
    
    y <- dplyr::bind_cols(
      data.frame(
      group = sub_df$group[boot],
      check.names = FALSE
      ),
      dplyr::select(sub_df, dplyr::all_of(Acapulco_cerebellar_regions_names))
       ) %>%
    dplyr::mutate(group = stats::relevel(group, ref = "HC")) %>%
    tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
        names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)
    
    nulls[j, ] <- y
  }
  nulls
  }
  
  res <- res %>% rlang::set_names(substances_names)
  return(res)
}

#### Perform it

In [18]:
dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)

# Save/load null distributions for substances vs control
nulls_fp_substances <- file.path("Statistics", "Linear_models", "Acapulco", "RData",
                 "Acapulco_nulls_substance_vs_control.RData")

if (!file.exists(nulls_fp_substances)) {
  Acapulco_nulls_substance_vs_control <- Acapulco_compute_null_distributions_substances(
  Acapulco_volumes,
  Acapulco_cerebellar_regions_names,
  substances_names,
  Acapulco_nulls_substance_vs_control,
  num_perm)

  # Ensure null matrices have region column names
  Acapulco_nulls_substance_vs_control <- purrr::imap(
    Acapulco_nulls_substance_vs_control,
    ~ { colnames(.x) <- Acapulco_cerebellar_regions_names; .x }
  )

  # Rebuild CIs per Substance x region from null distributions
  Acapulco_CIs <- purrr::imap_dfr(
    Acapulco_nulls_substance_vs_control,
    ~ {
      low  <- apply(.x, 2, quantile, probs = 0.025)
      high <- apply(.x, 2, quantile, probs = 0.975)
      tibble(
        Substance = .y,
        region    = names(low),
        perm_CI_lower    = as.numeric(low),
        perm_CI_upper     = as.numeric(high)
      )})

  # Compute permutation p-values from nulls and merge with CIs
  p_values_Acapulco_substance_vs_control <- Acapulco_lm_substance_vs_control %>%
    dplyr::select(-data) %>%
    dplyr::mutate(
      p_value = purrr::pmap_dbl(
        list(Substance, as.character(region), statistic),
        function(substance, roi, stat) {
          null_distribution <- Acapulco_nulls_substance_vs_control[[substance]][, roi]
          mean(abs(null_distribution) >= abs(stat))
        })
    ) %>%
    dplyr::group_by(Substance) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
    dplyr::ungroup() %>%
    dplyr::left_join(Acapulco_CIs, by = c("Substance", "region"))
    
  save(Acapulco_nulls_substance_vs_control,Acapulco_CIs,p_values_Acapulco_substance_vs_control, file = nulls_fp_substances)
} else {
  load(nulls_fp_substances)
}

## **1.2 Volume comparison by matching sample size**

### **1.2.1 Users**

#### **1.2.1.1 Set data and matchid**

Final file: **Acapulco_volumes_matchit_users_df**

In [19]:
Acapulco_cerebellar_regions_names <- Acapulco_volumes %>%
    select(contains(c(".","_Vol"))) %>% colnames()

Acapulco_volumes_matchit_users_raw <- Volumes_Cerebellum_Acapulco_raw %>% 
    mutate(Substance = case_when(Control != 0 ~ Substance, Control == 0 ~ "HC")) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(Sex = factor(Sex,levels = c("0","1")),
        group = if_else(Substance == "HC",0,1) %>%
                        factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")), .before = 2) %>%
    select(-c(Control))

# Match sample for each substance
Acapulco_volumes_matchit_model_users_raw <- Acapulco_volumes_matchit_users_raw %>% 
        matchit(formula = group ~ Age + Sex, 
                method = "nearest", distance = "glm", link = "probit",
                ratio = 1, replace = F)

Acapulco_volumes_matchit_users_df <- match.data(Acapulco_volumes_matchit_model_users_raw) %>% 
        select(-c(distance,weights,subclass)) %>% 
        mutate(GM_Vol = Total_Cerebel_Vol-Corpus.Medullare,
            Database = factor(Database))

#### **1.2.1.2 Harmonize and prepare data**

Final file: **Acapulco_volumes_matchit_users_harmonized_df**  
**Acapulco_volumes_matchit_users_harmonized_long**

In [20]:
# Harmonization volumes controling for Age, Sex
# Note: volumes are already corrected by eTIV

Acapulco_volumes_matchit_users_harmonized_df <- Acapulco_volumes_matchit_users_df %>% list() %>% 
  map(~ {
    df  <- .x %>% mutate(Database = factor(Database))
    roi <- intersect(Acapulco_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm) %>% select(-c(eTIV))
  }) %>% reduce(tibble)

[neuroCombat] Found 335 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 55 batches
[neuroCombat] Adjusting for  2  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data


In [21]:
# transform the dataset to long format
Acapulco_volumes_matchit_users_harmonized_long <- Acapulco_volumes_matchit_users_harmonized_df %>% 
    pivot_longer(names_to = "region", values_to = "volume", Acapulco_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))

#### **1.2.1.3 Linear models**

Final file: **p_values_Acapulco_matchit_users_vs_control**

##### Running the linear models

In [22]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

Acapulco_lm_matchit_users_vs_control <- Acapulco_volumes_matchit_users_harmonized_long %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        dat <- broom::tidy(m, conf.int = TRUE) %>%
            dplyr::filter(term == "groupUser") %>%
            dplyr::mutate(
                df = df.residual(m),
                sigma = sigma(m)
            )
        cont <- emmeans::emmeans(m, specs = "group") %>%
            emmeans::contrast(method = "pairwise")
        cont_df <- broom::tidy(cont)
        cohen_d <- cont_df$estimate / sigma(m)
        dat %>% mutate(effect.size = cohen_d)
    })) %>%
    unnest(test)

##### Obtain the null distributions

In [23]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("Acapulco_nulls_matchit_users_vs_control")) {
Acapulco_nulls_matchit_users_vs_control <- matrix(NA,ncol = Acapulco_volumes_matchit_users_harmonized_df %>% 
    select(Acapulco_cerebellar_regions_names) %>% ncol(),nrow = num_perm)
}

In [24]:
# Parallel permutation function (foreach) for users vs control

Acapulco_compute_null_distributions_matchit_users <- function(Acapulco_volumes_matchit_users_harmonized_df,
                              Acapulco_cerebellar_regions_names,
                              Acapulco_nulls_matchit_users_vs_control) {
  if (sum(is.na(Acapulco_nulls_matchit_users_vs_control)) > 0) {
  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)
  
  res <- foreach(i = seq_len(num_perm),
           .packages = c("dplyr", "tidyr", "purrr", "broom")) %dopar% {
    set.seed(42 + i)
    boot <- sample.int(n = nrow(Acapulco_volumes_matchit_users_harmonized_df), replace = TRUE)
    
    x <- dplyr::bind_cols(
    data.frame(
      group = Acapulco_volumes_matchit_users_harmonized_df$group[boot],
      check.names = FALSE
    ),
    dplyr::select(Acapulco_volumes_matchit_users_harmonized_df, dplyr::all_of(Acapulco_cerebellar_regions_names))
    ) %>%
    dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
    tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
              names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)
    
    x
  }
  
  for (j in seq_len(num_perm)) {
    Acapulco_nulls_matchit_users_vs_control[j, ] <- res[[j]]
  }
  }
  return(Acapulco_nulls_matchit_users_vs_control)
}


##### Perform it

In [25]:

dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for users vs control

nulls_fp_matchit_users <- file.path("Statistics", "Linear_models", "Acapulco", "RData",
                 "Acapulco_nulls_matchit_users_vs_control.RData")

if (!file.exists(nulls_fp_matchit_users)) {
    Acapulco_nulls_matchit_users_vs_control <- Acapulco_compute_null_distributions_matchit_users(
        Acapulco_volumes_matchit_users_harmonized_df,
        Acapulco_cerebellar_regions_names,
        Acapulco_nulls_matchit_users_vs_control)
    # add names to the nulls and transform to a tibble
    colnames(Acapulco_nulls_matchit_users_vs_control) <- as.character(Acapulco_lm_matchit_users_vs_control$region)
    
    ci_lower <- apply(Acapulco_nulls_matchit_users_vs_control, 2, quantile, probs = 0.025)  
    ci_upper <- apply(Acapulco_nulls_matchit_users_vs_control, 2, quantile, probs = 0.975)
    
    # Calculate permutation p-values aligned by region
    pvals <- purrr::map2_dbl(
        as.data.frame(Acapulco_nulls_matchit_users_vs_control),
        Acapulco_lm_matchit_users_vs_control$statistic, 
        ~ mean(abs(.x) >= abs(.y))
    )
    
    Acapulco_lm_matchit_users_vs_control_permuted <- Acapulco_lm_matchit_users_vs_control %>%
        select(-data) %>%
        mutate(
            p.value = pvals,
            perm_CI_lower = ci_lower[as.character(region)],
            perm_CI_upper = ci_upper[as.character(region)]
        )
    
    p_values_Acapulco_matchit_users_vs_control <- Acapulco_lm_matchit_users_vs_control_permuted %>%
        mutate(p.adj = p.adjust(p.value, method = "fdr"))
    
    save(Acapulco_nulls_matchit_users_vs_control,
        Acapulco_lm_matchit_users_vs_control_permuted,
        p_values_Acapulco_matchit_users_vs_control, 
        file = nulls_fp_matchit_users)
    } else {
    load(nulls_fp_matchit_users)
    }

### **1.2.2 Substances**

#### **1.2.2.1 Set data and matchid**

Final file: **Acapulco_volumes_matchit_substances_df**

In [26]:
Acapulco_cerebellar_regions_names <- Acapulco_volumes %>%
    select(contains(c(".","_Vol"))) %>% colnames()

# Define a vector with the names of the substance
substances_names  <- unique(Acapulco_volumes$Substance)[unique(Acapulco_volumes$Substance) !="HC"] %>% as.vector()
    
Acapulco_volumes_matchit_substances_raw <- Volumes_Cerebellum_Acapulco_raw %>% 
    mutate(Substance = case_when(Control != 0 ~ Substance, Control == 0 ~ "HC")) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(Sex = factor(Sex,levels = c("0","1")),
        group = if_else(Substance == "HC",0,1) %>%
                        factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")), .before = 2) %>%
    select(-c(Control))

set.seed(42)
# Prioritize HC from same Database; supplement with HC from other Databases if needed
Acapulco_volumes_matchit_model_substances_raw <- substances_names %>%
    purrr::map(~ {
        cases <- Acapulco_volumes_matchit_substances_raw %>% dplyr::filter(Substance == .x)
        dbs_cases <- unique(cases$Database)
        hc_same <- Acapulco_volumes_matchit_substances_raw %>% dplyr::filter(Substance == "HC",
                                                                                                                                                 Database %in% dbs_cases)
        needed <- nrow(cases) - nrow(hc_same)

        if (needed > 0) {
            hc_other_all <- Acapulco_volumes_matchit_substances_raw %>%
                dplyr::filter(Substance == "HC", !Database %in% dbs_cases)

            if (nrow(hc_other_all) == 0) {
                # fallback: use only same-db controls (truncate)
                controls_pool <- hc_same %>% dplyr::slice_sample(n = min(nrow(cases), nrow(hc_same)))
            } else {
                db_order <- hc_other_all %>%
                    dplyr::count(Database, name = "n") %>%
                    dplyr::arrange(dplyr::desc(n))

                cutoff_idx <- which(cumsum(db_order$n) >= needed)[1]
                chosen_dbs <- db_order$Database[seq_len(cutoff_idx)]

                hc_other_min <- hc_other_all %>% dplyr::filter(Database %in% chosen_dbs)

                # Sample exactly 'needed' from the minimal set of external databases
                hc_other_sample <- hc_other_min %>% dplyr::slice_sample(n = min(needed, nrow(hc_other_min)))

                controls_pool <- dplyr::bind_rows(hc_same, hc_other_sample)
            }
        } else {
            controls_pool <- hc_same %>% dplyr::slice_sample(n = nrow(cases))
        }

        analysis_df <- dplyr::bind_rows(cases, controls_pool)

        matchit(
            formula  = group ~ Age + Sex,
            data     = analysis_df,
            method   = "nearest",
            distance = "glm",
            link     = "probit",
            ratio    = 1,
            replace  = FALSE
        )
    }) %>%
    rlang::set_names(substances_names)

Acapulco_volumes_matchit_substances_df <- Acapulco_volumes_matchit_model_substances_raw %>%
    purrr::map(~ match.data(.x) %>%
        dplyr::select(-distance, -weights, -subclass) %>%
        dplyr::mutate(GM_Vol = Total_Cerebel_Vol-Corpus.Medullare,Database = factor(Database)))

#### **1.2.2.2 Harmonize and prepare data**

Final file:  
**Acapulco_volumes_matchit_substances_harmonized_df**   
**Acapulco_volumes_matchit_substances_harmonized_long**

In [27]:
# Harmonization volumes controling for Age, Sex
# Note: volumes are already corrected by eTIV

Acapulco_volumes_matchit_substances_harmonized_df <- Acapulco_volumes_matchit_substances_df %>% 
  map(~ {
    df  <- .x %>% mutate(Database = factor(Database))
    roi <- intersect(Acapulco_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm) %>% select(-c(eTIV))
  })

[neuroCombat] Found 321 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 25 batches
[neuroCombat] Adjusting for  2  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 3 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 6 batches
[neuroCombat] Adjusting for  2  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 10 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] 

In [28]:
# transform the dataset to long format
Acapulco_volumes_matchit_substances_harmonized_long <- Acapulco_volumes_matchit_substances_harmonized_df %>% 
    map(~ .x %>%
    pivot_longer(names_to = "region", values_to = "volume", Acapulco_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region))) %>%
    select(-c(Sex,Age)) )

#### **1.2.2.3 Linear models**

Final file: **p_values_Acapulco_matchit_substance_vs_control**

##### Running the linear models

In [29]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with HC as reference

Acapulco_lm_matchit_substance_vs_control <- map_df(
    substances_names,
    .f = ~ Acapulco_volumes_matchit_substances_harmonized_long[[.x]] %>%
        dplyr::filter(Substance %in% c("HC", .x)) %>%
        dplyr::select(group, region, volume) %>%
        tidyr::nest(data = -region) %>%
        dplyr::mutate(data = purrr::map(data, ~ dplyr::mutate(.x, group = stats::relevel(group, ref = "HC")))) %>%
        dplyr::mutate(test = purrr::map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% dplyr::mutate(effect.size = cohen_d)
        })) %>%
        tidyr::unnest(test) %>%
        dplyr::mutate(Substance = .x)
)

##### Obtain the null distributions

In [30]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions
if (!exists("Acapulco_nulls_matchit_substance_vs_control")){
    Acapulco_nulls_matchit_substance_vs_control <- lapply(1:length(substances_names),
        matrix, data = NA,
        nrow = num_perm, 
        ncol = Acapulco_volumes %>% select(Acapulco_cerebellar_regions_names) %>% ncol()
        ) %>%
    setNames(substances_names)
}

In [31]:
# Parallel permutation function (foreach) for substances vs control
Acapulco_compute_null_distributions_matchit_substances <- function(Acapulco_volumes,
                              Acapulco_cerebellar_regions_names,
                              substances_names,
                              Acapulco_nulls_matchit_substance_vs_control,
                              num_perm) {
  if (sum(is.na(unlist(Acapulco_nulls_matchit_substance_vs_control))) == 0) return(Acapulco_nulls_matchit_substance_vs_control)
  
  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)
  
  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr", "tidyr", "purrr", "broom")) %dopar% {
  # If this substance already has values, return as is
  current <- Acapulco_nulls_matchit_substance_vs_control[[i]]
  if (!is.null(current) && sum(is.na(current)) == 0) return(current)
  
  set.seed(42 + i)
  n_cols <- Acapulco_volumes %>% dplyr::select(dplyr::all_of(Acapulco_cerebellar_regions_names)) %>% ncol()
  nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)
  
  sub_df <- Acapulco_volumes %>%
    dplyr::filter(Substance %in% c("HC", substances_names[i])) %>%
    dplyr::select(Substance, group, dplyr::all_of(Acapulco_cerebellar_regions_names))
  
  for (j in seq_len(num_perm)) {
    boot <- sample.int(n = nrow(sub_df), replace = TRUE)
    
    y <- dplyr::bind_cols(
      data.frame(
      group = sub_df$group[boot],
      check.names = FALSE
      ),
      dplyr::select(sub_df, dplyr::all_of(Acapulco_cerebellar_regions_names))
    ) %>%
    dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
    tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
        names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)
    nulls[j, ] <- y
  }
  nulls
  }

  res <- res %>% rlang::set_names(substances_names)
  return(res)
}

##### Perform it

In [32]:

dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for substances vs control
nulls_fp_matchit_substances <- file.path("Statistics", "Linear_models", "Acapulco", "RData",
                 "Acapulco_nulls_matchit_substance_vs_control.RData")
if (!file.exists(nulls_fp_matchit_substances)) {
    Acapulco_nulls_matchit_substance_vs_control <- Acapulco_compute_null_distributions_matchit_substances(
    Acapulco_volumes,
    Acapulco_cerebellar_regions_names,
    substances_names,
    Acapulco_nulls_matchit_substance_vs_control,
    num_perm)
    # Ensure null matrices have region column names
    Acapulco_nulls_matchit_substance_vs_control <- purrr::imap(
      Acapulco_nulls_matchit_substance_vs_control,
      ~ { colnames(.x) <- Acapulco_cerebellar_regions_names; .x }
    )
    # Rebuild CIs per Substance x region from null distributions
    Acapulco_matchit_CIs <- purrr::imap_dfr(
      Acapulco_nulls_matchit_substance_vs_control,
      ~ {
        low  <- apply(.x, 2, quantile, probs = 0.025)
        high <- apply(.x, 2, quantile, probs = 0.975)
        tibble(
          Substance = .y,
          region    = names(low),
          perm_CI_lower    = as.numeric(low),
          perm_CI_upper     = as.numeric(high)
        )})
    # Compute permutation p-values from nulls and merge with CIs
    p_values_Acapulco_matchit_substance_vs_control <- Acapulco_lm_matchit_substance_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
        p_value = purrr::pmap_dbl(
          list(Substance, as.character(region), statistic),
          function(substance, roi, stat) {
            null_distribution <- Acapulco_nulls_matchit_substance_vs_control[[substance]][, roi]
            mean(abs(null_distribution) >= abs(stat))
          })
      ) %>%
      dplyr::group_by(Substance) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(Acapulco_matchit_CIs, by = c("Substance", "region"))

    save(Acapulco_nulls_matchit_substance_vs_control,Acapulco_matchit_CIs,
        p_values_Acapulco_matchit_substance_vs_control, file = nulls_fp_matchit_substances)
} else {
    load(nulls_fp_matchit_substances)
}

## **1.3 Volume comparison with permutation by k-fold cross-validation**

### **1.3.1 Exploring the databases**

plots: 
**plot_subs_per_DB**

In [33]:
# Count subjects per Database x Substance (including HC)
subjects_per_db_substance <- Volumes_Cerebellum_Acapulco_raw %>%
    mutate(Substance = if_else(Control == 0, "HC", as.character(Substance))) %>%
    filter(!is.na(Substance)) %>%
    mutate(Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC"))) %>%
    group_by(Database, Substance) %>%
    summarise(n_subjects = dplyr::n_distinct(ID_MRI), .groups = "drop") %>%
    arrange(Database, Substance)

plot_subs_per_DB <- subjects_per_db_substance %>%
    ggplot(aes(x = n_subjects,
                         y = forcats::fct_reorder(Database, n_subjects, .fun = sum, .desc = TRUE),
                         fill = Substance)) +
    geom_col(width = 0.7) +
    scale_fill_manual(values = pal_substances, drop = FALSE) +
    scale_x_continuous(expand = c(0, 0)) +
    labs(title = "Subjects per Database by Substance",
             x = "Number of subjects",
             y = "Database",
             fill = "Substance") +
    theme_output +
    theme(
      panel.grid.major.y = element_blank(),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 20)
    ) + coord_flip()

ggsave(plot = plot_subs_per_DB,filename = "Figures/Supplementaries/Figure_S1/Subjects_per_Database_by_Substance_Acapulco.png",
        width = 50, height=20, units="cm", bg = "white", dpi = 300)

### **1.3.2 splitting the k-folds**

Final file:  
**Acapulco_volumes_raw_folds_list**  

In [34]:
Acapulco_volumes_raw_folds_set <- Volumes_Cerebellum_Acapulco_raw %>%
    mutate(GM_Vol = Total_Cerebel_Vol-Corpus.Medullare)

# Leave-one-database-out per fold 
# Determine number of folds as number of unique databases
kfolds <- nlevels(Acapulco_volumes_raw_folds_set$Database)

# Leave-one-database-out cross-validation using rsample
Acapulco_volumes_raw_folds <- rsample::group_vfold_cv(
    data = Acapulco_volumes_raw_folds_set,
    group = Database,
    balance = "groups",
    v = kfolds
)

# For convenience, keep a copy named 'splits' to inspect
splits <- Acapulco_volumes_raw_folds

# Build training sets (analysis data) per fold for downstream steps
Acapulco_volumes_raw_folds_list <- purrr::map(
    Acapulco_volumes_raw_folds$splits,
    rsample::analysis
) %>% purrr::set_names(paste0("fold_", seq_len(kfolds)))

In [35]:
folds_names <- names(Acapulco_volumes_raw_folds_list)

### **1.3.3 Harmonize and prepare data**

Final file:  
**Acapulco_volumes_users_folds_harmonized**  
**Acapulco_volumes_users_folds_harmonized_long**

In [36]:
# Harmonization volumes controling for Age, Sex
# Note: volumes are already corrected by eTIV

Acapulco_volumes_folds_users_harmonized_df <- Acapulco_volumes_raw_folds_list %>%
  purrr::imap(~ {
    df  <- .x %>% dplyr::mutate(Database = factor(Database))
    roi <- intersect(Acapulco_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% tibble::as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm) %>%
      dplyr::select(-tidyselect::any_of(c("eTIV","Sex","Age")))
  }) %>% rlang::set_names(names(Acapulco_volumes_raw_folds_list))

[neuroCombat] Found 653 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 54 batches
[neuroCombat] Adjusting for  2  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 653 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 54 batches
[neuroCombat] Adjusting for  2  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 606 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroComb

In [37]:
Acapulco_volumes_users_folds_harmonized <- Acapulco_volumes_folds_users_harmonized_df %>% 
    map(~ .x %>%
    mutate(Substance = case_when(Control != 0 ~ Substance, Control == 0 ~ "HC")) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(group = if_else(Substance == "HC",0,1) %>% factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")), .before = 2) %>%
    select(-c(Control)) )

In [38]:
Acapulco_volumes_users_folds_harmonized_long <- map(Acapulco_volumes_users_folds_harmonized, function(df) {
    pivot_longer(df, names_to = "region", values_to = "volume", Acapulco_cerebellar_regions_names) %>%
        arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))
})

### **1.3.4 Running the analysis for each fold**

### **1.3.3.1 Users**

Final files:  
**p_values_Acapulco_folds_users_vs_control**  
**p_values_Acapulco_folds_users_vs_control_median**


#### Running the linear models

In [39]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

Acapulco_lm_folds_user_vs_control <- map_df(folds_names, function(fold) {
    Acapulco_volumes_users_folds_harmonized_long[[fold]] %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% mutate(effect.size = cohen_d)
        })) %>%
        unnest(test) %>%
        mutate(fold = fold)
})

#### Obtain the null distributions

In [40]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("Acapulco_nulls_folds_user_vs_control")) {
    n_cols <- length(Acapulco_cerebellar_regions_names)
    Acapulco_nulls_folds_user_vs_control <- lapply(seq_along(folds_names), function(i) {
        matrix(NA_real_, nrow = num_perm, ncol = n_cols)
    }) %>%
    setNames(folds_names)
}


In [41]:
# Parallel permutation function (foreach) for users vs control

Acapulco_compute_null_distributions_folds_users <- function(Acapulco_volumes_users_folds_harmonized,
                                                              Acapulco_cerebellar_regions_names,
                                                              folds_names,
                                                              Acapulco_nulls_folds_user_vs_control,
                                                              num_perm) {
    if (sum(is.na(unlist(Acapulco_nulls_folds_user_vs_control))) == 0) return(Acapulco_nulls_folds_user_vs_control)
    cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
    on.exit(parallel::stopCluster(cl), add = TRUE)
    doParallel::registerDoParallel(cl)

    res <- foreach(i = seq_along(folds_names),
                   .packages = c("dplyr","tidyr","purrr","broom")) %dopar% {
        fold_id <- folds_names[i]
        current <- Acapulco_nulls_folds_user_vs_control[[fold_id]]
        if (!is.null(current) && sum(is.na(current)) == 0) return(current)

        df <- Acapulco_volumes_users_folds_harmonized[[fold_id]] %>%
            dplyr::select(group, dplyr::all_of(Acapulco_cerebellar_regions_names))

        n_cols <- length(Acapulco_cerebellar_regions_names)
        nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

        for (j in seq_len(num_perm)) {
            set.seed(42 + i + j)
            boot <- sample.int(nrow(df), replace = TRUE)

            stats <- dplyr::bind_cols(
                    data.frame(group = df$group[boot], check.names = FALSE),
                    dplyr::select(df, dplyr::all_of(Acapulco_cerebellar_regions_names))
                ) %>%
                dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
                tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                                    names_to = "region", values_to = "volume") %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .x)
                    broom::tidy(m) %>%
                        dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
                        dplyr::select(statistic)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::pull(statistic)

            nulls[j, ] <- stats
        }
        nulls
    } %>% rlang::set_names(folds_names)

    return(res)
}

#### Perform it

In [42]:
dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for users vs control for each fold

nulls_fp_folds_users <- file.path("Statistics", "Linear_models", "Acapulco", "RData",
                 "Acapulco_nulls_folds_user_vs_control.RData")

if (!file.exists(nulls_fp_folds_users)) {
    Acapulco_nulls_folds_user_vs_control <- Acapulco_compute_null_distributions_folds_users(
        Acapulco_volumes_users_folds_harmonized,
        Acapulco_cerebellar_regions_names,
        folds_names,
        Acapulco_nulls_folds_user_vs_control,
        num_perm
        )

    # Ensure null matrices have region column names
    Acapulco_nulls_folds_user_vs_control <- purrr::imap(
      Acapulco_nulls_folds_user_vs_control,
      ~ { colnames(.x) <- Acapulco_cerebellar_regions_names; .x }
    )

    # Build CIs per fold x region from null distributions
    Acapulco_folds_CIs <- purrr::imap_dfr(
      Acapulco_nulls_folds_user_vs_control,
      ~ {
        low  <- apply(.x, 2, quantile, probs = 0.025)
        high <- apply(.x, 2, quantile, probs = 0.975)
        tibble::tibble(
          fold          = .y,
          region        = names(low),
          perm_CI_lower = as.numeric(low),
          perm_CI_upper = as.numeric(high)
        )
      }
    )

    # Compute permutation p-values per fold and merge with CIs
    p_values_Acapulco_folds_users_vs_control <- Acapulco_lm_folds_user_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
        p_value = purrr::pmap_dbl(
          list(fold, as.character(region), statistic),
          function(fold_id, roi, stat) {
            null_distribution <- Acapulco_nulls_folds_user_vs_control[[fold_id]][, roi]
            mean(abs(null_distribution) >= abs(stat))
          }
        )
      ) %>%
      dplyr::group_by(fold) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(Acapulco_folds_CIs, by = c("fold", "region"))

save(Acapulco_nulls_folds_user_vs_control,
        p_values_Acapulco_folds_users_vs_control,
        file = nulls_fp_folds_users)
} else {
load(nulls_fp_folds_users)
}


#### Obtain the mean, median, mode for all folds

In [43]:
Mode <- function(x) {
    ux <- unique(x)
    ux[which.max(tabulate(match(x, ux)))]
}

In [44]:
p_values_Acapulco_folds_users_vs_control_median <- p_values_Acapulco_folds_users_vs_control %>%
    group_by(region) %>%
    summarise(effect.size_median = median(effect.size,na.rm = TRUE),
        statistic_median = median(statistic, na.rm = TRUE),
        p_value_median = median(p_value, na.rm = TRUE),
        p_adj_median = median(p_adj, na.rm = TRUE),
        statistic_mode = Mode(statistic),
        effect.size_mode = Mode(effect.size),
        p_value_mode = Mode(p_value),
        p_adj_mode = Mode(p_adj),
        statistic_media = mean(statistic, na.rm = TRUE),
        effect.size_media = mean(effect.size, na.rm = TRUE),
        p_value_media = mean(p_value, na.rm = TRUE),
        p_adj_media = mean(p_adj, na.rm = TRUE),
        perm_CI_lower = mean(perm_CI_lower, na.rm = TRUE),
        perm_CI_upper = mean(perm_CI_upper, na.rm = TRUE),
        estimate_median = median(estimate, na.rm = TRUE),
        estimate_mode = Mode(estimate),
        estimate_media = mean(estimate, na.rm = TRUE)
        ) %>%
    ungroup()


### **1.3.3.2 Substances**

Final files:  
**p_values_Acapulco_folds_substance_vs_control**  
**p_values_Acapulco_folds_substance_vs_control_median**

#### Running the linear models

In [45]:
dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)

lm_fp_folds_substances <- file.path(
    "Statistics", "Linear_models", "Acapulco", "RData",
    "Acapulco_lm_folds_substance_vs_control.RData"
)

if (!file.exists(lm_fp_folds_substances)) {
    # Now that we've compared the HC groups lets do it for each HC vs each substance
    # Compute the group effect with a linear model (HC as reference)
    Acapulco_lm_folds_substance_vs_control <- purrr::map_df(substances_names, function(substance) {
        df_substance <- Acapulco_volumes_users_folds_harmonized_long %>%
            purrr::map(~ dplyr::filter(.x, Substance == "HC" | Substance == substance))

        purrr::map_df(names(df_substance), function(fold) {
            df_fold <- df_substance[[fold]]
            df_fold %>%
                dplyr::select(group, region, volume) %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(data = purrr::map(data, ~ dplyr::mutate(.x, group = relevel(group, ref = "HC")))) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .)
                    dat <- broom::tidy(m, conf.int = TRUE) %>%
                        dplyr::filter(term == "groupUser") %>%
                        dplyr::mutate(
                            df = df.residual(m),
                            sigma = sigma(m)
                        )
                    cont <- emmeans::emmeans(m, specs = "group") %>%
                        emmeans::contrast(method = "pairwise")
                    cont_df <- broom::tidy(cont)
                    cohen_d <- cont_df$estimate / sigma(m)
                    dat %>% dplyr::mutate(effect.size = cohen_d)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::mutate(
                    Substance = substance,
                    fold = fold
                )
        })
    })

    save(Acapulco_lm_folds_substance_vs_control, file = lm_fp_folds_substances)
} else {
    load(lm_fp_folds_substances)
}

#### Obtain the null distributions

In [46]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("Acapulco_nulls_folds_substance_vs_control")){
    n_cols <- length(Acapulco_cerebellar_regions_names)
    Acapulco_nulls_folds_substance_vs_control <- lapply(substances_names, function(substance) {
        lapply(seq_along(folds_names), function(i) {
            matrix(NA_real_, nrow = num_perm, ncol = n_cols)
        }) %>%
        setNames(folds_names)
    }) %>%
    setNames(substances_names)
}

In [47]:
# Parallel permutation function (foreach) for each substance vs control across folds

Acapulco_compute_null_distributions_folds_substances <- function(
  Acapulco_volumes_users_folds_harmonized,
  Acapulco_cerebellar_regions_names,
  substances_names,
  folds_names,
  Acapulco_nulls_folds_substance_vs_control,
  num_perm
) {
  if (sum(is.na(unlist(Acapulco_nulls_folds_substance_vs_control))) == 0) {
  return(Acapulco_nulls_folds_substance_vs_control)
  }

  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)

  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr","tidyr","purrr","broom")) %dopar% {
  substance <- substances_names[i]
  current_sub <- Acapulco_nulls_folds_substance_vs_control[[substance]]

  fold_results <- vector("list", length(folds_names))
  names(fold_results) <- folds_names

  for (f in seq_along(folds_names)) {
    fold_id <- folds_names[f]
    current <- if (!is.null(current_sub)) current_sub[[fold_id]] else NULL
    if (!is.null(current) && sum(is.na(current)) == 0) {
    fold_results[[fold_id]] <- current
    next
    }

    df <- Acapulco_volumes_users_folds_harmonized[[fold_id]] %>%
    dplyr::filter(Substance %in% c("HC", substance)) %>%
    dplyr::select(group, dplyr::all_of(Acapulco_cerebellar_regions_names))

    n_cols <- length(Acapulco_cerebellar_regions_names)
    nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

    for (j in seq_len(num_perm)) {
    set.seed(42 + i + f + j)
    boot <- sample.int(nrow(df), replace = TRUE)

    stats <- dplyr::bind_cols(
      data.frame(group = df$group[boot], check.names = FALSE),
      dplyr::select(df, dplyr::all_of(Acapulco_cerebellar_regions_names))
      ) %>%
      dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
      tidyr::pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                names_to = "region", values_to = "volume") %>%
      tidyr::nest(data = -region) %>%
      dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
        dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
        dplyr::select(statistic)
      })) %>%
      tidyr::unnest(test) %>%
      dplyr::pull(statistic)

    nulls[j, ] <- stats
    }
    fold_results[[fold_id]] <- nulls
  }
  fold_results
  } %>% rlang::set_names(substances_names)

  return(res)
}

#### Perform it

In [48]:

dir.create("Statistics/Linear_models/Acapulco/RData", showWarnings = FALSE, recursive = TRUE)
nulls_fp_folds_substances <- file.path("Statistics","Linear_models","Acapulco","RData",
                                       "Acapulco_nulls_folds_substance_vs_control.RData")

if (!file.exists(nulls_fp_folds_substances)) {
    Acapulco_nulls_folds_substance_vs_control <- Acapulco_compute_null_distributions_folds_substances(
        Acapulco_volumes_users_folds_harmonized,
        Acapulco_cerebellar_regions_names,
        substances_names,
        folds_names,
        Acapulco_nulls_folds_substance_vs_control,
        num_perm
    )

    # Build CIs per fold x region from null distributions
    Acapulco_folds_CIs_substances <- purrr::imap_dfr(
      Acapulco_nulls_folds_substance_vs_control,
      ~ {
        substance <- .y
        fold_list <- .x
        purrr::imap_dfr(fold_list, function(mat, fold_id) {
          low  <- apply(mat, 2, quantile, probs = 0.025)
          high <- apply(mat, 2, quantile, probs = 0.975)
          tibble::tibble(
            Substance     = substance,
            fold          = fold_id,
            region        = names(low),
            perm_CI_lower = as.numeric(low),
            perm_CI_upper = as.numeric(high)
          )
        })
      }
    )

    # Compute permutation p-values per fold and merge with CIs
    # Ensure null matrices have region column names
    Acapulco_nulls_folds_substance_vs_control <- purrr::imap(
      Acapulco_nulls_folds_substance_vs_control,
      ~ purrr::imap(.x, function(mat, fold_id) {
      colnames(mat) <- Acapulco_cerebellar_regions_names
      mat
      })
    )

    p_values_Acapulco_folds_substance_vs_control <- Acapulco_lm_folds_substance_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
      p_value = purrr::pmap_dbl(
        list(Substance, fold, as.character(region), statistic),
        function(substance, fold_id, roi, stat) {
        null_distribution <- Acapulco_nulls_folds_substance_vs_control[[substance]][[fold_id]][, roi]
        mean(abs(null_distribution) >= abs(stat))
        }
      )) %>%
      dplyr::group_by(Substance, fold) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(Acapulco_folds_CIs_substances, by = c("Substance", "fold"))

    save(Acapulco_nulls_folds_substance_vs_control,
       Acapulco_folds_CIs_substances,
       p_values_Acapulco_folds_substance_vs_control,
       file = nulls_fp_folds_substances)

} else {
    load(nulls_fp_folds_substances)
}


#### Obtain the mean, median and mode for all folds

In [49]:
p_values_Acapulco_folds_substance_vs_control_median <- p_values_Acapulco_folds_substance_vs_control %>%
    group_by(Substance, region) %>%
    summarise(effect.size_median = median(effect.size,na.rm = TRUE),
        statistic_median = median(statistic, na.rm = TRUE),
        p_value_median = median(p_value, na.rm = TRUE),
        p_adj_median = median(p_adj, na.rm = TRUE),
        statistic_mode = Mode(statistic),
        effect.size_mode = Mode(effect.size),
        p_value_mode = Mode(p_value),
        p_adj_mode = Mode(p_adj),
        statistic_media = mean(statistic, na.rm = TRUE),
        effect.size_media = mean(effect.size, na.rm = TRUE),
        p_value_media = mean(p_value, na.rm = TRUE),
        p_adj_media = mean(p_adj, na.rm = TRUE),
        perm_CI_lower = mean(perm_CI_lower, na.rm = TRUE),
        perm_CI_upper = mean(perm_CI_upper, na.rm = TRUE),
        estimate_median = median(estimate, na.rm = TRUE),
        estimate_mode = Mode(estimate),
        estimate_media = mean(estimate, na.rm = TRUE)
        ) %>%
    ungroup()



`summarise()` has grouped output by 'Substance'. You can override using the
`.groups` argument.


## **Save**

In [50]:
stats_fp <- "Statistics/Linear_models/Acapulco/Acapulco_volumes_statistics.RData"

if (!file.exists(stats_fp)) {
    save(Acapulco_volumes,
             Acapulco_volumes_long,
             p_values_Acapulco_user_vs_control,
             p_values_Acapulco_substance_vs_control,
             Acapulco_volumes_matchit_users_df,
             Acapulco_volumes_matchit_users_harmonized_df,
             Acapulco_volumes_matchit_users_harmonized_long,
             p_values_Acapulco_matchit_users_vs_control,
             Acapulco_volumes_matchit_substances_df,
             Acapulco_volumes_matchit_substances_harmonized_df,
             Acapulco_volumes_matchit_substances_harmonized_long,
             p_values_Acapulco_matchit_substance_vs_control,
             Acapulco_volumes_raw_folds_list,
             Acapulco_volumes_users_folds_harmonized,
             Acapulco_volumes_users_folds_harmonized_long,
             p_values_Acapulco_folds_users_vs_control,
             p_values_Acapulco_folds_users_vs_control_median,
             p_values_Acapulco_folds_substance_vs_control,
             p_values_Acapulco_folds_substance_vs_control_median,
             plot_subs_per_DB,
             file = stats_fp)
} else {
    message("File exists, skipping: ", stats_fp)
}

File exists, skipping: Statistics/Linear_models/Acapulco/Acapulco_volumes_statistics.RData



### Export matchid

In [51]:
dir.create("Data/matchid/", showWarnings = FALSE,recursive = TRUE)
Acapulco_volumes_matchit_substances_df$ALC %>% write_csv("Data/matchid/Acapulco_volumes_matchit.alc.csv")
Acapulco_volumes_matchit_substances_df$ATS %>% write_csv("Data/matchid/Acapulco_volumes_matchit.ats.csv")
Acapulco_volumes_matchit_substances_df$CANN %>% write_csv("Data/matchid/Acapulco_volumes_matchit.cann.csv")
Acapulco_volumes_matchit_substances_df$COC %>% write_csv("Data/matchid/Acapulco_volumes_matchit.coc.csv")
Acapulco_volumes_matchit_substances_df$NIC %>% write_csv("Data/matchid/Acapulco_volumes_matchit.nic.csv")
Acapulco_volumes_matchit_users_df %>% write_csv("Data/matchid/Acapulco_volumes_matchit.users.csv")

---
## **Sanity Check: Sensitivity analyses on complete-cases (CC), alternative-imputed (AI) datasets and Covariate-Adjusted (CA)**

The primary analyses above used Age and Sex imputed with `eTIV` as a predictor (potentially circular).  
These sensitivity analyses re-run the **User vs HC** and **Substance vs HC** linear models on:

- **CC** — subjects with fully observed Age and Sex (no imputation)
- **AI** — Age and Sex imputed using Substance group and Site only (no neuroanatomical predictors)
- **CA** - models fit with explicit covariate adjustment (**Age + Sex + eTIV**) in the linear model (instead of relying on imputation/harmonization choices)

*Note: Standard FDR-corrected p-values are used here (not permutation), which is appropriate for a sensitivity comparison.*


In [52]:

# Prepare CC and AI Acapulco volumes for sensitivity analyses
load("Data/RData/Sanity_check/Volumes_Cerebellum_Sanity_check.RData")

# Load CA (Covariate-Adjusted) data
load("Data/RData/Volumes_Cerebellum_Covariate_Adjusted.RData")

prepare_acapulco_sc <- function(harmonized_df) {
    harmonized_df %>%
        filter(Substance != "OPI") %>%
        mutate(
            group     = factor(if_else(Substance == "HC", 0, 1), labels = c("HC", "User")),
            Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","HC"))
        ) %>%
        mutate(group = factor(group, levels = c("User","HC"))) %>%
        mutate(Database = factor(Database),
               GM_Vol   = Total_Cerebel_Vol - Corpus.Medullare)
}

Acapulco_volumes_CC <- prepare_acapulco_sc(Volumes_Cerebellum_Acapulco_Harmonized_CC)
Acapulco_volumes_AI <- prepare_acapulco_sc(Volumes_Cerebellum_Acapulco_Harmonized_AI)
Acapulco_volumes_CA <- prepare_acapulco_sc(Volumes_Cerebellum_Acapulco_Harmonized_CA)

Acapulco_volumes_long_CC <- Acapulco_volumes_CC %>%
    pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                 names_to = "region", values_to = "volume") %>%
    arrange(region) %>%
    mutate(region = factor(region, levels = unique(.$region)))

Acapulco_volumes_long_AI <- Acapulco_volumes_AI %>%
    pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                 names_to = "region", values_to = "volume") %>%
    arrange(region) %>%
    mutate(region = factor(region, levels = unique(.$region)))

Acapulco_volumes_long_CA <- Acapulco_volumes_CA %>%
    pivot_longer(cols = dplyr::all_of(Acapulco_cerebellar_regions_names),
                 names_to = "region", values_to = "volume") %>%
    arrange(region) %>%
    mutate(region = factor(region, levels = unique(.$region)))


In [53]:

# ---- CC: User vs HC ----
p_values_Acapulco_CC_user_vs_control <- Acapulco_volumes_long_CC %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [54]:

# ---- CC: substance vs HC ----
substances_names_CC <- unique(Acapulco_volumes_CC$Substance)[
    unique(Acapulco_volumes_CC$Substance) != "HC"] %>% as.vector()

p_values_Acapulco_CC_substance_vs_control <- map_df(substances_names_CC, .f = ~ {
    Acapulco_volumes_long_CC %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [55]:

# ---- AI: User vs HC ----
p_values_Acapulco_AI_user_vs_control <- Acapulco_volumes_long_AI %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [56]:

# ---- AI: substance vs HC ----
substances_names_SC <- unique(Acapulco_volumes_AI$Substance)[
    unique(Acapulco_volumes_AI$Substance) != "HC"] %>% as.vector()

p_values_Acapulco_AI_substance_vs_control <- map_df(substances_names_SC, .f = ~ {
    Acapulco_volumes_long_AI %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [57]:
# ---- CA: User vs HC ----
p_values_Acapulco_CA_user_vs_control <- Acapulco_volumes_long_CA %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))

# ---- CA: substance vs HC ----
substances_names_CA <- unique(Acapulco_volumes_CA$Substance)[
    unique(Acapulco_volumes_CA$Substance) != "HC"] %>% as.vector()

p_values_Acapulco_CA_substance_vs_control <- map_df(substances_names_CA, .f = ~ {
    Acapulco_volumes_long_CA %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [58]:

# Save sanity check results
dir.create("Statistics/Sanity_check/Acapulco", recursive = TRUE, showWarnings = FALSE)
sc_fp <- "Statistics/Sanity_check/Acapulco/Acapulco_sanity_check_statistics.RData"

if (!file.exists(sc_fp)) {
    save(p_values_Acapulco_CC_user_vs_control,
         p_values_Acapulco_CC_substance_vs_control,
         p_values_Acapulco_AI_user_vs_control,
         p_values_Acapulco_AI_substance_vs_control,
         p_values_Acapulco_CA_user_vs_control,
         p_values_Acapulco_CA_substance_vs_control,
         file = sc_fp)
} else {
    message("File exists, skipping: ", sc_fp)
}

purrr::iwalk(
    list(
        Acapulco_CC_user_vs_control       = p_values_Acapulco_CC_user_vs_control,
        Acapulco_CC_substance_vs_control  = p_values_Acapulco_CC_substance_vs_control,
        Acapulco_AI_user_vs_control       = p_values_Acapulco_AI_user_vs_control,
        Acapulco_AI_substance_vs_control  = p_values_Acapulco_AI_substance_vs_control,
        Acapulco_CA_user_vs_control       = p_values_Acapulco_CA_user_vs_control,
        Acapulco_CA_substance_vs_control  = p_values_Acapulco_CA_substance_vs_control
    ),
    ~ write_csv(.x, file.path("Statistics/Sanity_check/Acapulco",
                               paste0(.y, ".csv")))
)